# List A + List B → List C — the clerk, shown working

**Audience:** compliance reviewers. No programming needed — run every cell top to bottom
(`Run All`) and read the tables.

**What you are looking at.** The clerk takes two inputs and produces one output:

| List | What it is | Where it comes from |
|------|-----------|---------------------|
| **List A** | The manual log — entries technicians typed | SharePoint (here: a sample table) |
| **List B** | Automatic detections | Seeq (here: a sample table) |
| **List C** | The resolved hourly verdict — valid / down / dropped, with the governing CFR paragraph and the records that produced it | **the clerk** |

Every scenario below shows its inputs, states the rule in plain language, runs the
**real production fold** (not a demo re-implementation — proven in the next cell), and
shows the result with a *why* column. Expected values come from the committed
**GOLDEN_TRAPS.md** answer key — hand-derived from the regulation, independent of the
clerk's own tests. **If the clerk disagrees with the key, the mismatch is flagged and
left visible.** A flagged mismatch is a finding, not a failure of the demo.

*No live connections: this notebook reads the sample data defined in its own cells and
`GOLDEN_TRAPS.md`, nothing else. No SharePoint, no PI, no Seeq.*

In [1]:
# Setup — imports, the golden answer key, and small display helpers.
import re
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

# The notebook lives in clerk_v2/; make sure the clerk package is importable.
NB_DIR = Path.cwd() if (Path.cwd() / "clerk").exists() else Path.cwd() / "clerk_v2"
sys.path.insert(0, str(NB_DIR))

# ---- Guardrail #1: the committed answer key must be present. ----
GOLDEN_PATH = NB_DIR / "GOLDEN_TRAPS.md"
if not GOLDEN_PATH.exists():
    raise SystemExit("GOLDEN_TRAPS.md is not in the repo — commit it first, then re-run. "
                     "This notebook does NOT reconstruct expected values from memory.")
GOLDEN_TEXT = GOLDEN_PATH.read_text()
print(f"Answer key: {GOLDEN_PATH}  ({len(GOLDEN_TEXT.splitlines())} lines)")

def golden_line(anchor):
    # Fetch the exact Expected line from the committed key; hard-fail if the
    # key no longer contains it (so a silently edited key cannot go unnoticed).
    if anchor not in GOLDEN_TEXT:
        raise AssertionError(f"GOLDEN_TRAPS.md no longer contains the anchor: {anchor!r}")
    return anchor

pd.set_option("display.max_colwidth", 120)

DAY = datetime(2026, 4, 1, tzinfo=timezone.utc)
def H(h, m=0):
    return DAY + timedelta(hours=h, minutes=m)
def hh(dt):
    return dt.strftime("%H:%M")

Answer key: /home/user/SeeQ-Flare-Project/clerk_v2/GOLDEN_TRAPS.md  (118 lines)


## The engine is the real one

The cell below imports the clerk's production fold and proves where it comes from.
`build_grid` is the function the nightly run itself calls (`clerk/run.py`); this notebook
adds **display only** — no calculation logic lives in this file.

In [2]:
import inspect

from clerk.fold import Status, fold
from clerk.grid import build_grid, source_down_hours
from clerk.listc import build_list_c
from clerk.schemas import (AnalyzerUnit, Capsule, CellValid, Event, EventType,
                           OperatingWindow, SiteConfig)

print("build_grid        <-", inspect.getsourcefile(build_grid))
print("source_down_hours <-", inspect.getsourcefile(source_down_hours))
print("fold              <-", inspect.getsourcefile(fold))
print("build_list_c      <-", inspect.getsourcefile(build_list_c))
assert inspect.getsourcefile(build_grid).endswith("clerk/grid.py"), "not the production module!"
assert inspect.getsourcefile(build_list_c).endswith("clerk/listc.py")
print("\nConfirmed: this is the production fold — the same code path as clerk/run.py.")

build_grid        <- /home/user/SeeQ-Flare-Project/clerk_v2/clerk/grid.py
source_down_hours <- /home/user/SeeQ-Flare-Project/clerk_v2/clerk/grid.py
fold              <- /home/user/SeeQ-Flare-Project/clerk_v2/clerk/fold.py
build_list_c      <- /home/user/SeeQ-Flare-Project/clerk_v2/clerk/listc.py

Confirmed: this is the production fold — the same code path as clerk/run.py.


In [3]:
# Input builders (List A rows -> TechEntry events; List B rows -> capsules)
# and the fold runner. Provenance note (task item 5): the fold tracks
# provenance NATIVELY — every List C row carries ContributingEventIDs citing
# the exact log entries (A-ids) and detections (CAP:... ids) that produced it.
# The display layer below only translates those ids into friendly labels.
_seq = iter(range(1, 999))

# The site's reason vocabulary -> governing CFR paragraph (same map as config.csv):
REASON_MAP = {"MM-01": "(i)", "NM-01": "(i)", "QA-01": "(iii)", "OK-01": "(i)", "UK-01": "(i)"}
REASON_WORDS = {"MM-01": "monitor malfunction", "NM-01": "non-monitor malfunction",
                "QA-01": "QA / calibration / maintenance", "OK-01": "other known", "UK-01": "unknown"}

def log_entry(analyzer, start, end, reason, note="", eid=None):
    eid = eid or f"A{next(_seq)}"
    return Event(EventID=eid, EventType=EventType.TechEntry, TargetEventID=None,
                 ExtentStartUTC=start, ExtentEndUTC=end, AnalyzerCEMIDs=[analyzer],
                 Category="", ReasonCode=reason, Actor="tech", ActedAt=start,
                 Reason=note, CorrectiveAction="", DetectionClass="")

def amendment(target, start, end, note=""):
    eid = f"{target.EventID}-AMD"
    return Event(EventID=eid, EventType=EventType.Correction, TargetEventID=target.EventID,
                 ExtentStartUTC=start, ExtentEndUTC=end, AnalyzerCEMIDs=list(target.AnalyzerCEMIDs),
                 Category="", ReasonCode=target.ReasonCode, Actor="tech",
                 ActedAt=target.ActedAt + timedelta(hours=1), Reason=note,
                 CorrectiveAction="", DetectionClass="")

def detection(analyzer, start, end, cls="status-offline"):
    return Capsule(Analyzer=analyzer, DetectionClass=cls,
                   CapsuleStartUTC=start, CapsuleEndUTC=end)

def detection_ticket(analyzer, start, end, reason=None, cls="status-offline"):
    # A detection as an EVENT (so a reason can attach via Confirmation).
    det = Event(EventID=f"B{next(_seq)}", EventType=EventType.SeeqDetection,
                TargetEventID=None, ExtentStartUTC=start, ExtentEndUTC=end,
                AnalyzerCEMIDs=[analyzer], Category="", ReasonCode="",
                Actor="seeq-auto", ActedAt=start, Reason="", CorrectiveAction="",
                DetectionClass=cls)
    out = [det]
    if reason:
        out.append(Event(EventID=f"{det.EventID}-CF", EventType=EventType.Confirmation,
                         TargetEventID=det.EventID, ExtentStartUTC=start, ExtentEndUTC=end,
                         AnalyzerCEMIDs=[analyzer], Category="", ReasonCode=reason,
                         Actor="tech", ActedAt=start + timedelta(minutes=1), Reason="",
                         CorrectiveAction="", DetectionClass=""))
    return out

def dispute(target, why):
    base = [Event(EventID=f"{target.EventID}-DIS", EventType=EventType.DismissalProposed,
                  TargetEventID=target.EventID, ExtentStartUTC=target.ExtentStartUTC,
                  ExtentEndUTC=target.ExtentEndUTC, AnalyzerCEMIDs=list(target.AnalyzerCEMIDs),
                  Category="dismissal", ReasonCode="", Actor="tech",
                  ActedAt=target.ActedAt + timedelta(minutes=30), Reason=why,
                  CorrectiveAction="", DetectionClass=""),
            Event(EventID=f"{target.EventID}-APP", EventType=EventType.Approval,
                  TargetEventID=target.EventID, ExtentStartUTC=target.ExtentStartUTC,
                  ExtentEndUTC=target.ExtentEndUTC, AnalyzerCEMIDs=list(target.AnalyzerCEMIDs),
                  Category="dismissal", ReasonCode="", Actor="supervisor",
                  ActedAt=target.ActedAt + timedelta(minutes=60), Reason="Approved",
                  CorrectiveAction="", DetectionClass="")]
    return base

def unit(analyzer, unit_name, basis="", role="not-diluent-corrected", species="", in_service=None):
    return AnalyzerUnit(analyzer, unit_name, SeeqCovered=True, DiluentsRole=role,
                        DiluentSpecies=species, DiluentBasis=basis,
                        InServiceDateUTC=in_service)

RUNS = []                 # (scenario, events, capsules, cells) — feeds the
_CURRENT = {"s": "?"}     # enriched List C export at the end of the notebook

def run_fold(events, capsules, units, operating, start, end):
    config = SiteConfig(SiteTimeZoneIANA="America/New_York", LookbackMonths=8,
                        LateXThresholdDays=7, JitterToleranceMin=5,
                        PartialOperatingHourApplicability={},
                        ReasonParagraphMap=dict(REASON_MAP))
    cells = build_grid(events=list(events), capsules=list(capsules),
                       operating_windows=list(operating), analyzer_units=list(units),
                       qa_windows=[], config=config, window_start=start, window_end=end)
    RUNS.append((_CURRENT["s"], list(events), list(capsules), cells))
    return cells

In [4]:
# Display helpers: input tables, the List C "why" table, and the golden check.
RULE_WORDS = {
    "(i)":        "full operating hour — needs a valid point in each 15-min quadrant",
    "(ii)":       "partial operating hour — needs a valid point in each OPERATED quadrant",
    "(iii)(A)":   "maintenance/QA hour — needs two valid points >= 15 minutes apart",
    "(iii)(B)":   "single-quadrant hour — needs at least one valid point",
    "(iv)":       "daily validation failed — hour invalid unless a later same-hour pass recovers it",
    "not-operating": "unit offline — recorded, excluded from availability math entirely",
}
VERDICT = {CellValid.valid: "VALID", CellValid.invalid: "DOWN",
           CellValid.not_operating: "DROPPED (unit offline)"}

ALL_CELLS, RESULTS = [], []          # for the global invariant + final scoreboard
CSV_A, CSV_B, CSV_C = [], [], []     # exported at the end as list_a/b/c.csv

def show_a(scenario, events):
    _CURRENT["s"] = scenario
    rows = [{"Entry": e.EventID,
             "Kind": {"TechEntry": "log entry", "Correction": "AMENDMENT",
                      "DismissalProposed": "DISPUTE", "Approval": "dispute approved",
                      "SeeqDetection": "detection ticket", "Confirmation": "confirmation",
                      "WindowPick": "WINNER PICK (synthetic approval)"}[e.EventType.value],
             "Analyzer": e.AnalyzerCEMIDs[0],
             "From": hh(e.ExtentStartUTC) if e.ExtentStartUTC else "",
             "To": hh(e.ExtentEndUTC) if e.ExtentEndUTC else "(none)",
             "Reason": (f"{e.ReasonCode} — {REASON_WORDS[e.ReasonCode]}" if e.ReasonCode else e.Reason)}
            for e in events]
    for e in events:
        CSV_A.append({"Scenario": scenario, "EventID": e.EventID, "Type": e.EventType.value,
                      "Analyzer": e.AnalyzerCEMIDs[0],
                      "StartUTC": e.ExtentStartUTC.isoformat() if e.ExtentStartUTC else "",
                      "EndUTC": e.ExtentEndUTC.isoformat() if e.ExtentEndUTC else "",
                      "ReasonCode": e.ReasonCode, "Note": e.Reason})
    return pd.DataFrame(rows)

def show_b(scenario, capsules):
    rows = [{"Detection": f"B-cap{i}", "Analyzer": c.Analyzer, "Class": c.DetectionClass,
             "From": hh(c.CapsuleStartUTC), "To": hh(c.CapsuleEndUTC)}
            for i, c in enumerate(capsules, 1)]
    for c in capsules:
        CSV_B.append({"Scenario": scenario, "Analyzer": c.Analyzer,
                      "DetectionClass": c.DetectionClass,
                      "CapsuleStartUTC": c.CapsuleStartUTC.isoformat(),
                      "CapsuleEndUTC": c.CapsuleEndUTC.isoformat()})
    return pd.DataFrame(rows)

def show_c(scenario, cells, only_analyzers=None):
    ALL_CELLS.extend(cells)
    rows = []
    for c in cells:
        if only_analyzers and c.Analyzer not in only_analyzers:
            continue
        prov = "; ".join("detection " + p.split(":")[1] + " " + p.split(":")[3][11:16]
                         + "-" + p.split(":")[6][11:16] if p.startswith("CAP:")
                         else "entry " + p for p in c.ContributingEventIDs) or "—"
        rows.append({"Analyzer": c.Analyzer, "Hour (UTC)": hh(c.HourStartUTC),
                     "Hour (site)": c.HourLocalLabel[-9:],
                     "Verdict": VERDICT[c.Valid], "Paragraph": c.RuleApplied,
                     "Why": RULE_WORDS.get(c.RuleApplied, c.RuleApplied),
                     "Based on": prov})
        CSV_C.append({"Scenario": scenario, "Analyzer": c.Analyzer,
                      "HourStartUTC": c.HourStartUTC.isoformat(),
                      "Verdict": VERDICT[c.Valid], "Paragraph": c.RuleApplied,
                      "ContributingRecords": ";".join(c.ContributingEventIDs)})
    return pd.DataFrame(rows)

def down_hours(cells, analyzer):
    return sorted(hh(c.HourStartUTC) for c in cells
                  if c.Analyzer == analyzer and c.Valid is CellValid.invalid)

def check(name, actual, expected, authority, anchor=None):
    if anchor:
        golden_line(anchor)   # prove the committed key still says this
    ok = actual == expected
    RESULTS.append({"Scenario": name, "Expected": str(expected), "Actual": str(actual),
                    "Match": "MATCH" if ok else "*** MISMATCH ***", "Authority": authority})
    flag = "OK  " if ok else ">>> MISMATCH — flagged as a finding, sample NOT adjusted <<<\n    "
    print(f"{flag}{name}\n    expected {expected}  |  actual {actual}")
    return ok

---
## Scenario 1 — `A1 + B1 = C1` (log and detection agree)

**The rule in plain language:** when a technician's log entry and an automatic detection
describe the same outage, List C shows **one** down hour — not two — and cites **both**
records as its basis. A malfunction hour is judged under paragraph **(i)**: a full
operating hour needs a valid data point in each 15-minute quadrant; an hour fully
consumed by the outage has none, so it is down.

In [5]:
A = [log_entry("NOX-01", H(9), H(10), "MM-01", "Analyzer offline — blown fuse")]
B = [detection("NOX-01", H(9), H(10))]
display(show_a("S1", A)); display(show_b("S1", B))

C = run_fold(A, B, [unit("NOX-01", "U-S1")],
             [OperatingWindow("U-S1", H(8), H(11))], H(8), H(11))
display(show_c("S1", C))
check("S1  log + detection agree -> one down hour, both cited",
      down_hours(C, "NOX-01"), ["09:00"],
      "reg (i) — derived, not a numbered trap")
print("Takeaway: agreement produces ONE verdict with TWO supporting records — no double-counting.")

,Entry,Kind,Analyzer,From,To,Reason
0,A1,log entry,NOX-01,09:00,10:00,MM-01 — monitor malfunction


,Detection,Analyzer,Class,From,To
0,B-cap1,NOX-01,status-offline,09:00,10:00


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOX-01,09:00,05:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry A1; detection NOX-01 09-
2,NOX-01,10:00,06:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—


OK  S1  log + detection agree -> one down hour, both cited
    expected ['09:00']  |  actual ['09:00']
Takeaway: agreement produces ONE verdict with TWO supporting records — no double-counting.


---
## Scenario 2 — `A1 + A2 + B1 = C1` (two logs plus a detection)

**Rule:** multiple log entries and a detection covering the same window still resolve to
one verdict per hour. Here both logs carry reason `QA-01` (maintenance/QA), so the hour is
judged under paragraph **(iii)** — two valid points ≥ 15 minutes apart. The whole hour is
consumed, so no valid points exist: down.

In [6]:
A = [log_entry("NOX-01", H(9), H(9, 30), "QA-01", "CGA run — first half"),
     log_entry("NOX-01", H(9, 30), H(10), "QA-01", "CGA run — second half")]
B = [detection("NOX-01", H(9), H(10))]
display(show_a("S2", A)); display(show_b("S2", B))

C = run_fold(A, B, [unit("NOX-01", "U-S2")],
             [OperatingWindow("U-S2", H(8), H(11))], H(8), H(11))
display(show_c("S2", C))
check("S2  two logs + detection -> one down hour under (iii)",
      down_hours(C, "NOX-01"), ["09:00"],
      "reg (iii) — derived, not a numbered trap")
print("Takeaway: three input records, one resolved hour — List C never double-counts overlap.")

,Entry,Kind,Analyzer,From,To,Reason
0,A2,log entry,NOX-01,09:00,09:30,QA-01 — QA / calibration / maintenance
1,A3,log entry,NOX-01,09:30,10:00,QA-01 — QA / calibration / maintenance


,Detection,Analyzer,Class,From,To
0,B-cap1,NOX-01,status-offline,09:00,10:00


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOX-01,09:00,05:00 EDT,DOWN,(iii)(A),maintenance/QA hour — needs two valid points >= 15 minutes apart,entry A2; entry A3; detection NOX-01 09-
2,NOX-01,10:00,06:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—


OK  S2  two logs + detection -> one down hour under (iii)
    expected ['09:00']  |  actual ['09:00']
Takeaway: three input records, one resolved hour — List C never double-counts overlap.


---
## Scenario 3 — `B1 + B2 + A1 = C1` (two detections merging with one log)

**Rule:** Seeq often reports one physical outage as several back-to-back capsules. The
fold unions them (with the log) into the hour's single verdict — the seam between B1 and
B2 at 09:30 does not split the outage or double it.

In [7]:
A = [log_entry("NOX-01", H(9), H(10), "MM-01", "Sample pump failure")]
B = [detection("NOX-01", H(9), H(9, 30)),        # abutting pair:
     detection("NOX-01", H(9, 30), H(10))]       # end == next start
display(show_a("S3", A)); display(show_b("S3", B))

C = run_fold(A, B, [unit("NOX-01", "U-S3")],
             [OperatingWindow("U-S3", H(8), H(11))], H(8), H(11))
display(show_c("S3", C))
check("S3  two abutting detections + log -> one down hour",
      down_hours(C, "NOX-01"), ["09:00"],
      "reg (i) — derived; merge behavior per fold union")
print("Takeaway: fragmented detections merge — one outage, one down hour, all three records cited.")

,Entry,Kind,Analyzer,From,To,Reason
0,A4,log entry,NOX-01,09:00,10:00,MM-01 — monitor malfunction


,Detection,Analyzer,Class,From,To
0,B-cap1,NOX-01,status-offline,09:00,09:30
1,B-cap2,NOX-01,status-offline,09:30,10:00


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOX-01,09:00,05:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry A4; detection NOX-01 09-; detection NOX-01 09-
2,NOX-01,10:00,06:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—


OK  S3  two abutting detections + log -> one down hour
    expected ['09:00']  |  actual ['09:00']
Takeaway: fragmented detections merge — one outage, one down hour, all three records cited.


---
## Scenario 4 — Amendment: `A1 → A1′ + B1 = C1`

**Rule:** a technician logs 09:00–12:00, then amends it to 09:00–10:00. List C reflects
the **corrected** extent (only 09:00 down); the original entry and the amendment both
remain visible — the ledger is append-only, nothing is erased.

In [8]:
a1 = log_entry("NOX-01", H(9), H(12), "MM-01", "Analyzer down (initial estimate)")
a1_amend = amendment(a1, H(9), H(10), "Corrected: restored by 10:00, not 12:00")
A = [a1, a1_amend]
B = [detection("NOX-01", H(9), H(10))]
display(show_a("S4", A)); display(show_b("S4", B))

C = run_fold(A, B, [unit("NOX-01", "U-S4")],
             [OperatingWindow("U-S4", H(8), H(13))], H(8), H(13))
display(show_c("S4", C))

# Both A1 and A1' stay visible in the folded record:
obs = fold(A)[0]
display(pd.DataFrame([{"Folded record": obs.origin_event_id,
                       "Current extent": f"{hh(obs.extent_start_utc)}-{hh(obs.extent_end_utc)}",
                       "History (all versions kept)": " , ".join(obs.contributing_event_ids)}]))
check("S4  amendment -> C reflects corrected extent (1 down hour, not 3)",
      down_hours(C, "NOX-01"), ["09:00"],
      "fold doctrine — append-only ledger, corrections applied on replay")
print("Takeaway: C follows the amendment; hours 10:00-11:00 are valid again; A1 and A1' both remain on file.")

,Entry,Kind,Analyzer,From,To,Reason
0,A5,log entry,NOX-01,09:00,12:00,MM-01 — monitor malfunction
1,A5-AMD,AMENDMENT,NOX-01,09:00,10:00,MM-01 — monitor malfunction


,Detection,Analyzer,Class,From,To
0,B-cap1,NOX-01,status-offline,09:00,10:00


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOX-01,09:00,05:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry A5; detection NOX-01 09-
2,NOX-01,10:00,06:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
3,NOX-01,11:00,07:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
4,NOX-01,12:00,08:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—


,Folded record,Current extent,History (all versions kept)
0,A5,09:00-10:00,"A5 , A5-AMD"


OK  S4  amendment -> C reflects corrected extent (1 down hour, not 3)
    expected ['09:00']  |  actual ['09:00']
Takeaway: C follows the amendment; hours 10:00-11:00 are valid again; A1 and A1' both remain on file.


---
## Scenario 5 — Dismissal: `B1 + A1(dispute) = C1`

**Rule:** a detection can be **dismissed** with a reason (here: a sensor purge that looked
like an outage). Once the dismissal is approved *and* the live detection still corroborates
the signed window, the hours come back. List C **excludes** the dismissed time — but the
dismissed record stays visible below **with its reason**, so nothing disappears silently.

In [9]:
b1_ticket = detection_ticket("NOX-01", H(9), H(10))          # the detection as a ticket
a1_dispute = dispute(b1_ticket[0], "Sensor purge cycle — not a real outage")
A = b1_ticket + a1_dispute
B = [detection("NOX-01", H(9), H(10))]                        # the live capsule
display(show_a("S5", A)); display(show_b("S5", B))

C = run_fold(A, B, [unit("NOX-01", "U-S5")],
             [OperatingWindow("U-S5", H(8), H(11))], H(8), H(11))
display(show_c("S5", C))

dismissed = [o for o in fold(A) if o.status is Status.dismissed]
display(pd.DataFrame([{"Dismissed record": o.origin_event_id,
                       "Window": f"{hh(o.extent_start_utc)}-{hh(o.extent_end_utc)}",
                       "Status": o.status.value,
                       "Reason given": next(e.Reason for e in A
                                            if e.EventID == f"{o.origin_event_id}-DIS")}
                      for o in dismissed]))
check("S5  approved dismissal -> hour restored to VALID",
      down_hours(C, "NOX-01"), [],
      "workflow doctrine — signed dismissal + corroborating live capsule")
print("Takeaway: C excludes the dismissed hour, and the dismissal itself stays on file with its reason.")

,Entry,Kind,Analyzer,From,To,Reason
0,B6,detection ticket,NOX-01,09:00,10:00,
1,B6-DIS,DISPUTE,NOX-01,09:00,10:00,Sensor purge cycle — not a real outage
2,B6-APP,dispute approved,NOX-01,09:00,10:00,Approved


,Detection,Analyzer,Class,From,To
0,B-cap1,NOX-01,status-offline,09:00,10:00


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOX-01,09:00,05:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,detection NOX-01 09-
2,NOX-01,10:00,06:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—


,Dismissed record,Window,Status,Reason given
0,B6,09:00-10:00,Dismissed,Sensor purge cycle — not a real outage


OK  S5  approved dismissal -> hour restored to VALID
    expected []  |  actual []
Takeaway: C excludes the dismissed hour, and the dismissal itself stays on file with its reason.


---
## Scenario 6 — Golden trap T2: maintenance vs fault (same window, different reason)

**Rule:** the SAME invalid window 08:22–10:40 is judged differently depending on why it
happened. `QA-01` (maintenance) → paragraph **(iii)**: hours 08:00 and 10:00 keep enough
valid time (22 and 20 minutes) to count. `MM-01` (fault) → paragraph **(i)**: any consumed
quadrant downs the hour. **Same data: 1 down hour vs 3.**

In [10]:
print("GOLDEN_TRAPS.md T2a:", golden_line("**Expected down = {09:00}** (1 hour)"))
print("GOLDEN_TRAPS.md T2b:", golden_line("**Expected down = {08:00, 09:00, 10:00}** (3 hours)"))

for reason, expected, label in [("QA-01", ["09:00"], "T2a maintenance"),
                                 ("MM-01", ["08:00", "09:00", "10:00"], "T2b fault")]:
    A = detection_ticket("NOX-01", H(8, 22), H(10, 40), reason=reason)
    display(show_a("S6-" + reason, A))
    C = run_fold(A, [], [unit("NOX-01", "U-S6")],
                 [OperatingWindow("U-S6", H(8), H(11))], H(8), H(11))
    display(show_c("S6-" + reason, C))
    check(f"S6  {label} ({reason}) 08:22-10:40", down_hours(C, "NOX-01"), expected,
          "GOLDEN_TRAPS.md T2 — verbatim reg authority")
print("Takeaway: the reason code changes the governing paragraph — 1 vs 3 down hours on identical data.")

GOLDEN_TRAPS.md T2a: **Expected down = {09:00}** (1 hour)
GOLDEN_TRAPS.md T2b: **Expected down = {08:00, 09:00, 10:00}** (3 hours)


,Entry,Kind,Analyzer,From,To,Reason
0,B7,detection ticket,NOX-01,08:22,10:40,
1,B7-CF,confirmation,NOX-01,08:22,10:40,QA-01 — QA / calibration / maintenance


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,VALID,(iii)(A),maintenance/QA hour — needs two valid points >= 15 minutes apart,entry B7
1,NOX-01,09:00,05:00 EDT,DOWN,(iii)(A),maintenance/QA hour — needs two valid points >= 15 minutes apart,entry B7
2,NOX-01,10:00,06:00 EDT,VALID,(iii)(A),maintenance/QA hour — needs two valid points >= 15 minutes apart,entry B7


OK  S6  T2a maintenance (QA-01) 08:22-10:40
    expected ['09:00']  |  actual ['09:00']


,Entry,Kind,Analyzer,From,To,Reason
0,B8,detection ticket,NOX-01,08:22,10:40,
1,B8-CF,confirmation,NOX-01,08:22,10:40,MM-01 — monitor malfunction


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B8
1,NOX-01,09:00,05:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B8
2,NOX-01,10:00,06:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B8


OK  S6  T2b fault (MM-01) 08:22-10:40
    expected ['08:00', '09:00', '10:00']  |  actual ['08:00', '09:00', '10:00']
Takeaway: the reason code changes the governing paragraph — 1 vs 3 down hours on identical data.


---
## Scenario 7 — Golden trap T4: diluent propagation (and the T4d finding)

**Rule:** NOx and CO are *corrected using* the O2 monitor. When O2 is down, NOx and CO
cannot be computed — their hours go down **even though their own signals are clean**.
Propagation is one-way (a NOx outage never downs O2) and — per the key's **T4d** — must
work whether the O2 outage was **detected** or **logged manually** (doctrine D7).

In [11]:
print("GOLDEN_TRAPS.md T4:", golden_line("NOx: {09:00, 10:00, 13:00}"))
B15 = [unit("NOx", "B15", basis="O2", role="diluent-corrected", species="O2"),
       unit("O2", "B15", role="diluent", species="O2"),
       unit("CO", "B15", basis="O2", role="diluent-corrected", species="O2"),
       unit("TEMP-degF", "B15")]
OPER = [OperatingWindow("B15", H(8), H(15))]

# T4 (detected): O2 outage 09:00-11:00 via detection; NOx's own outage 13:00-14:00.
A = detection_ticket("O2", H(9), H(11), reason="MM-01") + \
    detection_ticket("NOx", H(13), H(14), reason="MM-01")
display(show_a("S7-T4", A))
C = run_fold(A, [], B15, OPER, H(8), H(15))
display(show_c("S7-T4", C, only_analyzers={"NOx", "O2", "CO", "TEMP-degF"}))
check("S7  T4 NOx (propagated 09-10 + own 13)", down_hours(C, "NOx"),
      ["09:00", "10:00", "13:00"], "GOLDEN_TRAPS.md T4 — App-F/doctrine authority")
check("S7  T4 CO (propagated)", down_hours(C, "CO"), ["09:00", "10:00"],
      "GOLDEN_TRAPS.md T4")
check("S7  T4 TEMP-degF (not corrected -> unaffected)", down_hours(C, "TEMP-degF"), [],
      "GOLDEN_TRAPS.md T4")
check("S7  T4 one-way (O2 NOT down at 13:00)", down_hours(C, "O2"), ["09:00", "10:00"],
      "GOLDEN_TRAPS.md T4")

GOLDEN_TRAPS.md T4: NOx: {09:00, 10:00, 13:00}


,Entry,Kind,Analyzer,From,To,Reason
0,B9,detection ticket,O2,09:00,11:00,
1,B9-CF,confirmation,O2,09:00,11:00,MM-01 — monitor malfunction
2,B10,detection ticket,NOx,13:00,14:00,
3,B10-CF,confirmation,NOx,13:00,14:00,MM-01 — monitor malfunction


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOx,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOx,09:00,05:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,—
2,NOx,10:00,06:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,—
3,NOx,11:00,07:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
4,NOx,12:00,08:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
5,NOx,13:00,09:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B10
6,NOx,14:00,10:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
7,O2,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
8,O2,09:00,05:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B9
9,O2,10:00,06:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B9


OK  S7  T4 NOx (propagated 09-10 + own 13)
    expected ['09:00', '10:00', '13:00']  |  actual ['09:00', '10:00', '13:00']
OK  S7  T4 CO (propagated)
    expected ['09:00', '10:00']  |  actual ['09:00', '10:00']
OK  S7  T4 TEMP-degF (not corrected -> unaffected)
    expected []  |  actual []
OK  S7  T4 one-way (O2 NOT down at 13:00)
    expected ['09:00', '10:00']  |  actual ['09:00', '10:00']


True

In [12]:
# T4d — the SAME O2 outage, but logged MANUALLY (no detection at all).
print("GOLDEN_TRAPS.md T4d:", golden_line("Expected identical: NOx {09,10,+own}, CO {09,10}"))
A = [log_entry("O2", H(9), H(11), "MM-01", "O2 analyzer down — logged by tech, no capsule")] + \
    detection_ticket("NOx", H(13), H(14), reason="MM-01")
display(show_a("S7-T4d", A))
C = run_fold(A, [], B15, OPER, H(8), H(15))
display(show_c("S7-T4d", C, only_analyzers={"NOx", "O2", "CO"}))
check("S7  T4d manual O2 outage still propagates to NOx", down_hours(C, "NOx"),
      ["09:00", "10:00", "13:00"], "GOLDEN_TRAPS.md T4d — doctrine D7 (source-agnostic)")
check("S7  T4d manual O2 outage still propagates to CO", down_hours(C, "CO"),
      ["09:00", "10:00"], "GOLDEN_TRAPS.md T4d")

GOLDEN_TRAPS.md T4d: Expected identical: NOx {09,10,+own}, CO {09,10}


,Entry,Kind,Analyzer,From,To,Reason
0,A11,log entry,O2,09:00,11:00,MM-01 — monitor malfunction
1,B12,detection ticket,NOx,13:00,14:00,
2,B12-CF,confirmation,NOx,13:00,14:00,MM-01 — monitor malfunction


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOx,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOx,09:00,05:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
2,NOx,10:00,06:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
3,NOx,11:00,07:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
4,NOx,12:00,08:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
5,NOx,13:00,09:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B12
6,NOx,14:00,10:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
7,O2,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
8,O2,09:00,05:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry A11
9,O2,10:00,06:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry A11


>>> MISMATCH — flagged as a finding, sample NOT adjusted <<<
    S7  T4d manual O2 outage still propagates to NOx
    expected ['09:00', '10:00', '13:00']  |  actual ['13:00']
>>> MISMATCH — flagged as a finding, sample NOT adjusted <<<
    S7  T4d manual O2 outage still propagates to CO
    expected ['09:00', '10:00']  |  actual []


False

---
## Scenario 8 — Golden trap T3a: unit offline = dropped, not down

**Rule:** an hour when the **unit itself wasn't running** is not a monitoring failure — it
is *dropped*: recorded, but excluded from availability math entirely (out of both the
numerator and the denominator). Only the online hour of this outage counts as down.

In [13]:
print("GOLDEN_TRAPS.md T3a:", golden_line("T3a → down {10:00}, dropped {09:00}"))
A = detection_ticket("NOX-01", H(9), H(11), reason="MM-01")
display(show_a("S8", A))
C = run_fold(A, [], [unit("NOX-01", "U-S8")],
             [OperatingWindow("U-S8", H(8), H(9)),      # offline 09:00-10:00
              OperatingWindow("U-S8", H(10), H(11))], H(8), H(11))
display(show_c("S8", C))
dropped = sorted(hh(c.HourStartUTC) for c in C if c.Valid is CellValid.not_operating)
check("S8  T3a down hours", down_hours(C, "NOX-01"), ["10:00"],
      "GOLDEN_TRAPS.md T3 — verbatim reg authority")
check("S8  T3a dropped hours", dropped, ["09:00"], "GOLDEN_TRAPS.md T3")
print("Takeaway: offline hours are dropped with a reason — they neither inflate nor hide downtime.")

GOLDEN_TRAPS.md T3a: T3a → down {10:00}, dropped {09:00}


,Entry,Kind,Analyzer,From,To,Reason
0,B13,detection ticket,NOX-01,09:00,11:00,
1,B13-CF,confirmation,NOX-01,09:00,11:00,MM-01 — monitor malfunction


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOX-01,08:00,04:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
1,NOX-01,09:00,05:00 EDT,DROPPED (unit offline),not-operating,"unit offline — recorded, excluded from availability math entirely",entry B13
2,NOX-01,10:00,06:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B13


OK  S8  T3a down hours
    expected ['10:00']  |  actual ['10:00']
OK  S8  T3a dropped hours
    expected ['09:00']  |  actual ['09:00']
Takeaway: offline hours are dropped with a reason — they neither inflate nor hide downtime.


---
## Scenario 9 — Golden trap T5a: per-obligation rollup

**Rule:** source FCC carries a **NOx obligation** (permanent NOx-P + temp NOx-T) and a
**separate O2 obligation** (O2-F). The source is "down" for NOx only when **every NOx
monitor** is down at once — and the always-valid **O2-F must not participate**: a valid
monitor for one pollutant can never cover another pollutant's outage (doctrine D8).

In [14]:
print("GOLDEN_TRAPS.md T5a:", golden_line("**Expected source-down = {12:00, 13:00}.**"))
FCC = [unit("NOx-P", "FCC"), unit("NOx-T", "FCC"),
       unit("O2-F", "FCC", role="diluent", species="O2")]
A = detection_ticket("NOx-P", H(10), H(14), reason="MM-01") + \
    detection_ticket("NOx-T", H(12), H(16), reason="MM-01")
display(show_a("S9", A))
C = run_fold(A, [], FCC, [OperatingWindow("FCC", H(10), H(16))], H(10), H(16))
display(show_c("S9", C, only_analyzers={"NOx-P", "NOx-T"}))

# Monitor-level downtime is ALWAYS reported (global invariant):
check("S9  monitor-level NOx-P", down_hours(C, "NOx-P"),
      ["10:00", "11:00", "12:00", "13:00"], "GOLDEN_TRAPS.md T5 (monitor level)")
check("S9  monitor-level NOx-T", down_hours(C, "NOx-T"),
      ["12:00", "13:00", "14:00", "15:00"], "GOLDEN_TRAPS.md T5 (monitor level)")

# The rollup AS THE ENGINE COMPUTES IT TODAY — intersecting every monitor on
# the unit, including the valid O2-F:
engine_rollup = {u: sorted(hh(h) for h in hs)
                 for u, hs in source_down_hours(FCC, C).items()}
check("S9  T5a source rollup (engine, per-unit incl. O2-F)",
      engine_rollup.get("FCC", []), ["12:00", "13:00"],
      "GOLDEN_TRAPS.md T5a — App-F/doctrine D8 authority")

# Display-layer demonstration of the per-OBLIGATION grouping the key requires
# (same production function, scoped to the NOx obligation's monitors only):
nox_obligation = [u for u in FCC if u.Analyzer in ("NOx-P", "NOx-T")]
obligation_rollup = {u: sorted(hh(h) for h in hs)
                     for u, hs in source_down_hours(nox_obligation, C).items()}
print("Per-obligation grouping (NOx monitors only):", obligation_rollup.get("FCC", []))

GOLDEN_TRAPS.md T5a: **Expected source-down = {12:00, 13:00}.**


,Entry,Kind,Analyzer,From,To,Reason
0,B14,detection ticket,NOx-P,10:00,14:00,
1,B14-CF,confirmation,NOx-P,10:00,14:00,MM-01 — monitor malfunction
2,B15,detection ticket,NOx-T,12:00,16:00,
3,B15-CF,confirmation,NOx-T,12:00,16:00,MM-01 — monitor malfunction


,Analyzer,Hour (UTC),Hour (site),Verdict,Paragraph,Why,Based on
0,NOx-P,10:00,06:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B14
1,NOx-P,11:00,07:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B14
2,NOx-P,12:00,08:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B14
3,NOx-P,13:00,09:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B14
4,NOx-P,14:00,10:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
5,NOx-P,15:00,11:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
6,NOx-T,10:00,06:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
7,NOx-T,11:00,07:00 EDT,VALID,(i),full operating hour — needs a valid point in each 15-min quadrant,—
8,NOx-T,12:00,08:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B15
9,NOx-T,13:00,09:00 EDT,DOWN,(i),full operating hour — needs a valid point in each 15-min quadrant,entry B15


OK  S9  monitor-level NOx-P
    expected ['10:00', '11:00', '12:00', '13:00']  |  actual ['10:00', '11:00', '12:00', '13:00']
OK  S9  monitor-level NOx-T
    expected ['12:00', '13:00', '14:00', '15:00']  |  actual ['12:00', '13:00', '14:00', '15:00']
>>> MISMATCH — flagged as a finding, sample NOT adjusted <<<
    S9  T5a source rollup (engine, per-unit incl. O2-F)
    expected ['12:00', '13:00']  |  actual []
Per-obligation grouping (NOx monitors only): ['12:00', '13:00']


---
## Scenario 10 — the enriched List C record (the standalone compliance record)

Everything above showed List C at **hourly** resolution. The clerk also resolves each
downtime into **one record** carrying everything a reviewer needs without joining tables:
the extent actually used, the reason/note/corrective action from the log, **which source
window won** (A, B, or a picked value), the **approver name + decision + timestamp**, the
governing paragraph, the hourly detail, and full provenance.

Record states: **auto-approved** (A and B concur — approver "auto"), **pending** (A and B
disagree, nobody has decided), **approved** (a winner-pick event exists, with the name).

> **Synthetic approvals.** No approval screen or workflow exists yet. The `WINNER PICK`
> events below are synthetic input events standing in for that future step — the fold
> reads them exactly the way it already reads dismissal events.

In [15]:
# Render enriched records as a table.
def fmt_w(ws):
    return " + ".join(f"{s:%H:%M}-{e:%H:%M}" for s, e in ws) or "—"

def show_records(scenario, records):
    return pd.DataFrame([{
        "Analyzer": r.Analyzer, "State": r.State,
        "Resolved extent": fmt_w(r.ResolvedWindows), "Min": r.ResolvedMinutes,
        "Source used": r.SourceUsed,
        "A window": fmt_w(r.WindowA), "B window(s)": fmt_w(r.WindowsB),
        "Reason": r.ReasonCode, "Note": r.Note, "Corrective action": r.CorrectiveAction,
        "Approver": r.ApproverName or "—", "Decision": r.ApproverDecision or "(pending)",
        "Approved at": r.ApprovedAtUTC.strftime("%m-%d %H:%M") if r.ApprovedAtUTC else "—",
        "Paragraphs": ";".join(r.GoverningParagraphs),
        "Down hours": len(r.DownHours),
        "Disagreement": r.Disagreement or "—",
    } for r in records])

def pick_event(target, choice, approver, start=None, end=None):
    # SYNTHETIC approval stand-in — no UI/workflow exists yet; this event
    # is what the future approval step will append to the ledger.
    return Event(EventID=f"{target.EventID}-PICK", EventType=EventType.WindowPick,
                 TargetEventID=target.EventID, ExtentStartUTC=start, ExtentEndUTC=end,
                 AnalyzerCEMIDs=list(target.AnalyzerCEMIDs), Category=choice,
                 ReasonCode="", Actor=approver, ActedAt=target.ActedAt + timedelta(hours=3),
                 Reason="synthetic approval stand-in", CorrectiveAction="", DetectionClass="")

### 10a — concurrence: log and detection agree → auto-approved
The tech's log and Seeq's detection state the same window, so the record approves itself:
approver `auto`, decision `concurrence`, and the note + corrective action ride along.

In [16]:
a1 = log_entry("NOX-01", H(8), H(10), "MM-01", "Blown fuse on sample pump")
a1.CorrectiveAction = "Fuse replaced, verified on span gas 10:05"
A = [a1]; B = [detection("NOX-01", H(8), H(10))]
display(show_a("S10a", A)); display(show_b("S10a", B))
C = run_fold(A, B, [unit("NOX-01", "U-S10")], [OperatingWindow("U-S10", H(7), H(11))], H(7), H(11))
rec = build_list_c(A, B, C)
display(show_records("S10a", rec))
check("S10a concurrence -> auto-approved, approver 'auto'",
      (rec[0].State, rec[0].ApproverName), ("auto-approved", "auto"),
      "task #52 — record state model")

,Entry,Kind,Analyzer,From,To,Reason
0,A16,log entry,NOX-01,08:00,10:00,MM-01 — monitor malfunction


,Detection,Analyzer,Class,From,To
0,B-cap1,NOX-01,status-offline,08:00,10:00


,Analyzer,State,Resolved extent,Min,Source used,A window,B window(s),Reason,Note,Corrective action,Approver,Decision,Approved at,Paragraphs,Down hours,Disagreement
0,NOX-01,auto-approved,08:00-10:00,120.0,concurrence (A=B),08:00-10:00,08:00-10:00,MM-01,Blown fuse on sample pump,"Fuse replaced, verified on span gas 10:05",auto,concurrence,—,(i),2,—


OK  S10a concurrence -> auto-approved, approver 'auto'
    expected ('auto-approved', 'auto')  |  actual ('auto-approved', 'auto')


True

### 10b — THE reconciliation case, unresolved → pending
Seeq detects **08:00–09:00** and **09:15–10:00** — the signal was **valid 09:00–09:15**.
The tech logged one block **08:00–10:00**. The record must keep both claims, call out the
gap, and sit in `pending` until a human decides. *(While pending, the resolved extent is
the union both-claims-stand rule the hourly grid already uses — flagged as a provisional
policy in `clerk/listc.py`.)*

In [17]:
tech_log = log_entry("NOX-01", H(8), H(10), "MM-01", "Analyzer down 8-10 (tech estimate)")
A = [tech_log]
B = [detection("NOX-01", H(8), H(9)), detection("NOX-01", H(9, 15), H(10))]
display(show_a("S10b", A)); display(show_b("S10b", B))
C = run_fold(A, B, [unit("NOX-01", "U-S10")], [OperatingWindow("U-S10", H(7), H(11))], H(7), H(11))
rec = build_list_c(A, B, C)
display(show_records("S10b", rec))
r = rec[0]
check("S10b gap case unresolved -> state pending", r.State, "pending",
      "task #5 reconciliation")
check("S10b Seeq's two intervals preserved (gap NOT collapsed)",
      fmt_w(r.WindowsB), "08:00-09:00 + 09:15-10:00", "task #5 reconciliation")
check("S10b the 09:00-09:15 valid gap is called out",
      "09:00-09:15" in r.Disagreement and "VALID" in r.Disagreement, True,
      "task #5 reconciliation")

,Entry,Kind,Analyzer,From,To,Reason
0,A17,log entry,NOX-01,08:00,10:00,MM-01 — monitor malfunction


,Detection,Analyzer,Class,From,To
0,B-cap1,NOX-01,status-offline,08:00,09:00
1,B-cap2,NOX-01,status-offline,09:15,10:00


,Analyzer,State,Resolved extent,Min,Source used,A window,B window(s),Reason,Note,Corrective action,Approver,Decision,Approved at,Paragraphs,Down hours,Disagreement
0,NOX-01,pending,08:00-10:00,120.0,union (pending),08:00-10:00,08:00-09:00 + 09:15-10:00,MM-01,Analyzer down 8-10 (tech estimate),,—,(pending),—,(i),2,A logs 08:00-10:00; B detects 08:00-09:00 + 09:15-10:00; B shows signal VALID during 09:00-09:15


OK  S10b gap case unresolved -> state pending
    expected pending  |  actual pending
OK  S10b Seeq's two intervals preserved (gap NOT collapsed)
    expected 08:00-09:00 + 09:15-10:00  |  actual 08:00-09:00 + 09:15-10:00
OK  S10b the 09:00-09:15 valid gap is called out
    expected True  |  actual True


True

### 10c — resolved by picking Seeq's windows
A synthetic winner-pick (`use-B`, approver named) folds in: the record approves, the
resolved extent becomes Seeq's two intervals, and the valid gap is honored —
**105 minutes, not 120**. Note B's raw claim and the disagreement stay on the record.

In [18]:
A2 = A + [pick_event(tech_log, "use-B", "r.huddleston")]
display(show_a("S10c", A2))
C = run_fold(A2, B, [unit("NOX-01", "U-S10")], [OperatingWindow("U-S10", H(7), H(11))], H(7), H(11))
rec = build_list_c(A2, B, C)
display(show_records("S10c", rec))
r = rec[0]
check("S10c pick-Seeq -> gap restored (105 min), approver named",
      (r.State, r.ResolvedMinutes, r.ApproverName),
      ("approved", 105.0, "r.huddleston"), "task #5 reconciliation")

,Entry,Kind,Analyzer,From,To,Reason
0,A17,log entry,NOX-01,08:00,10:00,MM-01 — monitor malfunction
1,A17-PICK,WINNER PICK (synthetic approval),NOX-01,,(none),synthetic approval stand-in


,Analyzer,State,Resolved extent,Min,Source used,A window,B window(s),Reason,Note,Corrective action,Approver,Decision,Approved at,Paragraphs,Down hours,Disagreement
0,NOX-01,approved,08:00-09:00 + 09:15-10:00,105.0,pick:use-B,08:00-10:00,08:00-09:00 + 09:15-10:00,MM-01,Analyzer down 8-10 (tech estimate),,r.huddleston,pick:use-B,04-01 11:00,(i),2,A logs 08:00-10:00; B detects 08:00-09:00 + 09:15-10:00; B shows signal VALID during 09:00-09:15


OK  S10c pick-Seeq -> gap restored (105 min), approver named
    expected ('approved', 105.0, 'r.huddleston')  |  actual ('approved', 105.0, 'r.huddleston')


True

### 10d — resolved by picking the tech's block
Only an **explicit human pick** may null the 09:00–09:15 window — and when it does, the
record shows exactly who decided, what they decided, and when; Seeq's intervals remain
visible on the record rather than being erased.

In [19]:
A3 = A + [pick_event(tech_log, "use-A", "r.huddleston")]
display(show_a("S10d", A3))
C = run_fold(A3, B, [unit("NOX-01", "U-S10")], [OperatingWindow("U-S10", H(7), H(11))], H(7), H(11))
rec = build_list_c(A3, B, C)
display(show_records("S10d", rec))
r = rec[0]
check("S10d pick-tech -> gap nulled EXPLICITLY (120 min), B still on record",
      (r.State, r.ResolvedMinutes, fmt_w(r.WindowsB)),
      ("approved", 120.0, "08:00-09:00 + 09:15-10:00"), "task #5 reconciliation")

# Guardrail: the pick resolved the RECORD — hourly verdicts are identical
# with and without it (validity math untouched).
base = [(c.HourStartUTC, c.Valid) for c in run_fold(A, B, [unit("NOX-01", "U-S10")],
        [OperatingWindow("U-S10", H(7), H(11))], H(7), H(11))]
picked = [(c.HourStartUTC, c.Valid) for c in run_fold(A3, B, [unit("NOX-01", "U-S10")],
          [OperatingWindow("U-S10", H(7), H(11))], H(7), H(11))]
RUNS.pop(); RUNS.pop()   # verification re-runs, not scenarios — keep the export clean
check("S10 guardrail: picks never change hourly verdicts", base == picked, True,
      "task #52 constraint — validity math unchanged")

,Entry,Kind,Analyzer,From,To,Reason
0,A17,log entry,NOX-01,08:00,10:00,MM-01 — monitor malfunction
1,A17-PICK,WINNER PICK (synthetic approval),NOX-01,,(none),synthetic approval stand-in


,Analyzer,State,Resolved extent,Min,Source used,A window,B window(s),Reason,Note,Corrective action,Approver,Decision,Approved at,Paragraphs,Down hours,Disagreement
0,NOX-01,approved,08:00-10:00,120.0,pick:use-A,08:00-10:00,08:00-09:00 + 09:15-10:00,MM-01,Analyzer down 8-10 (tech estimate),,r.huddleston,pick:use-A,04-01 11:00,(i),2,A logs 08:00-10:00; B detects 08:00-09:00 + 09:15-10:00; B shows signal VALID during 09:00-09:15


OK  S10d pick-tech -> gap nulled EXPLICITLY (120 min), B still on record
    expected ('approved', 120.0, '08:00-09:00 + 09:15-10:00')  |  actual ('approved', 120.0, '08:00-09:00 + 09:15-10:00')
OK  S10 guardrail: picks never change hourly verdicts
    expected True  |  actual True


True

---
## Scoreboard, findings, and the exported files

Everything below is generated from the runs above. A `MISMATCH` row is a **finding about
the engine**, deliberately left visible — the sample was not adjusted to hide it.

In [20]:
# Global invariant (GOLDEN_TRAPS.md): no hour anywhere in the set is 'not assessed'.
unassessed = [c for c in ALL_CELLS if c.Valid is CellValid.not_assessed]
check("Global invariant: zero not_assessed hours across all scenarios",
      len(unassessed), 0, "GOLDEN_TRAPS.md global invariants")

scoreboard = pd.DataFrame(RESULTS)
display(scoreboard)

mismatches = scoreboard[scoreboard.Match != "MATCH"]
if len(mismatches):
    print(f"\n{len(mismatches)} FINDING(S) — engine does not yet match the answer key:")
    for _, r in mismatches.iterrows():
        print(f"  - {r.Scenario}\n      expected {r.Expected}  |  actual {r.Actual}  [{r.Authority}]")
else:
    print("\nAll scenarios match the committed answer key.")

OK  Global invariant: zero not_assessed hours across all scenarios
    expected 0  |  actual 0


,Scenario,Expected,Actual,Match,Authority
0,"S1 log + detection agree -> one down hour, both cited",['09:00'],['09:00'],MATCH,"reg (i) — derived, not a numbered trap"
1,S2 two logs + detection -> one down hour under (iii),['09:00'],['09:00'],MATCH,"reg (iii) — derived, not a numbered trap"
2,S3 two abutting detections + log -> one down hour,['09:00'],['09:00'],MATCH,reg (i) — derived; merge behavior per fold union
3,"S4 amendment -> C reflects corrected extent (1 down hour, not 3)",['09:00'],['09:00'],MATCH,"fold doctrine — append-only ledger, corrections applied on replay"
4,S5 approved dismissal -> hour restored to VALID,[],[],MATCH,workflow doctrine — signed dismissal + corroborating live capsule
5,S6 T2a maintenance (QA-01) 08:22-10:40,['09:00'],['09:00'],MATCH,GOLDEN_TRAPS.md T2 — verbatim reg authority
6,S6 T2b fault (MM-01) 08:22-10:40,"['08:00', '09:00', '10:00']","['08:00', '09:00', '10:00']",MATCH,GOLDEN_TRAPS.md T2 — verbatim reg authority
7,S7 T4 NOx (propagated 09-10 + own 13),"['09:00', '10:00', '13:00']","['09:00', '10:00', '13:00']",MATCH,GOLDEN_TRAPS.md T4 — App-F/doctrine authority
8,S7 T4 CO (propagated),"['09:00', '10:00']","['09:00', '10:00']",MATCH,GOLDEN_TRAPS.md T4
9,S7 T4 TEMP-degF (not corrected -> unaffected),[],[],MATCH,GOLDEN_TRAPS.md T4



3 FINDING(S) — engine does not yet match the answer key:
  - S7  T4d manual O2 outage still propagates to NOx
      expected ['09:00', '10:00', '13:00']  |  actual ['13:00']  [GOLDEN_TRAPS.md T4d — doctrine D7 (source-agnostic)]
  - S7  T4d manual O2 outage still propagates to CO
      expected ['09:00', '10:00']  |  actual []  [GOLDEN_TRAPS.md T4d]
  - S9  T5a source rollup (engine, per-unit incl. O2-F)
      expected ['12:00', '13:00']  |  actual []  [GOLDEN_TRAPS.md T5a — App-F/doctrine D8 authority]


In [21]:
# Write the exported files and finish.
# list_c.csv is now the ENRICHED record — the standalone compliance record;
# the hourly verdict grid keeps its resolution in list_c_hours.csv.
out = NB_DIR
pd.DataFrame(CSV_A).to_csv(out / "list_a.csv", index=False)
pd.DataFrame(CSV_B).to_csv(out / "list_b.csv", index=False)
pd.DataFrame(CSV_C).to_csv(out / "list_c_hours.csv", index=False)

enriched_rows = []
for scenario, ev, caps, cells in RUNS:
    for r in build_list_c(ev, caps, cells):
        enriched_rows.append({
            "Scenario": scenario, "Analyzer": r.Analyzer, "State": r.State,
            "ResolvedWindows": fmt_w(r.ResolvedWindows), "ResolvedMinutes": r.ResolvedMinutes,
            "SourceUsed": r.SourceUsed, "Disagreement": r.Disagreement,
            "WindowA": fmt_w(r.WindowA), "WindowsB": fmt_w(r.WindowsB),
            "ReasonCode": r.ReasonCode, "Note": r.Note,
            "CorrectiveAction": r.CorrectiveAction, "ApproverName": r.ApproverName,
            "ApproverDecision": r.ApproverDecision,
            "ApprovedAtUTC": r.ApprovedAtUTC.isoformat() if r.ApprovedAtUTC else "",
            "GoverningParagraphs": ";".join(r.GoverningParagraphs),
            "DownHours": ";".join(h.strftime("%H:%M") for h in r.DownHours),
            "ContributingRecords": ";".join(r.ContributingRecords),
        })
pd.DataFrame(enriched_rows).to_csv(out / "list_c.csv", index=False)

print("Wrote:", out / "list_a.csv")
print("Wrote:", out / "list_b.csv")
print("Wrote:", out / "list_c.csv", f"(ENRICHED records: {len(enriched_rows)} rows)")
print("Wrote:", out / "list_c_hours.csv", f"(hourly verdicts: {len(CSV_C)} rows)")
golden_rows = scoreboard[scoreboard.Authority.str.startswith("GOLDEN")]
print(f"\nGolden-trap checks: {int((golden_rows.Match == 'MATCH').sum())} matched, "
      f"{int((golden_rows.Match != 'MATCH').sum())} known mismatches (still flagged).")

Wrote: /home/user/SeeQ-Flare-Project/clerk_v2/list_a.csv
Wrote: /home/user/SeeQ-Flare-Project/clerk_v2/list_b.csv
Wrote: /home/user/SeeQ-Flare-Project/clerk_v2/list_c.csv (ENRICHED records: 18 rows)
Wrote: /home/user/SeeQ-Flare-Project/clerk_v2/list_c_hours.csv (hourly verdicts: 87 rows)

Golden-trap checks: 11 matched, 3 known mismatches (still flagged).
